## Environmental programming using Python

## Assignment topic: Suitability Mapping of Nature-Based Solutions Locations to Tackle Hydroclimatic Extremes and Water Quality Degradation Using Machine Learning 

Group 4: Elias Zgheib, Ndra Malky, Rashmi Krishnamurthy, Teju Kumar Nagaraju

This notebook utilizes the previously clipped raster (from Task-1) to extract the pixel values for all data parameters and save it to the dataframe (csv). Pixel values are extracted for pixels lying within the defined boundary

# Task 2A — Extract Pixel Values (Inside Boundary) to CSV


## 1) Library imports

In [1]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

## 2) User inputs (Edit this cell)

In [2]:
folders = {
    "AET":  r"E:\VUB\Final\AET_Clipped",
    "LULC": r"E:\VUB\Final\LULC_Clipped",
    "P":    r"E:\VUB\Final\Precipitation_Clipped",
    "RZSM": r"E:\VUB\Final\RootZoneSoilMoisture_Clipped",
    "TEMP": r"E:\VUB\Final\Temperature_Mean_Clipped",
}

SOIL_FILE = r"E:\VUB\Final\Soil_Clipped\Soil_HSG_10km_clipped.tif"

YEARS = range(2014, 2025)  # 2014..2023
OUT_CSV = r"E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv"

CATEGORICAL = {"LULC", "SOIL"}
ZERO_AS_NODATA = {"LULC", "SOIL"}

print("Folders:")
for k, v in folders.items():
    print(f"  {k}: {v}")
print("SOIL_FILE:", SOIL_FILE)
print("YEARS:", list(YEARS)[:3], "...", list(YEARS)[-3:])
print("OUT_CSV:", OUT_CSV)


Folders:
  AET: E:\VUB\Final\AET_Clipped
  LULC: E:\VUB\Final\LULC_Clipped
  P: E:\VUB\Final\Precipitation_Clipped
  RZSM: E:\VUB\Final\RootZoneSoilMoisture_Clipped
  TEMP: E:\VUB\Final\Temperature_Mean_Clipped
SOIL_FILE: E:\VUB\Final\Soil_Clipped\Soil_HSG_10km_clipped.tif
YEARS: [2014, 2015, 2016] ... [2022, 2023, 2024]
OUT_CSV: E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv


## 3) Missing data checks
Generative AI tool was used to generate this part of code

In [ ]:
missing = False

for var, folder in folders.items():
    if not os.path.isdir(folder):
        print("Missing folder:", var, "->", folder)
        missing = True
    else:
        tifs = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]
        print(f" {var}: {len(tifs)} tif(s) in {folder}")

if not os.path.exists(SOIL_FILE):
    print("Missing SOIL_FILE:", SOIL_FILE)
    missing = True
else:
    print("SOIL_FILE exists")

out_dir = os.path.dirname(OUT_CSV)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
    print("Output directory ready:", out_dir)

if missing:
    raise FileNotFoundError("Fix missing paths in the Config cell and rerun.")


## 4) Helper functions
Generative AI tool was used to generate this part of code

In [ ]:
def find_year_tif(folder, year):
    y = str(year)
    for f in os.listdir(folder):
        if f.lower().endswith(".tif") and y in f:
            return os.path.join(folder, f)
    return None

def window_coords_pixelid(ref_ds, win):
    W = ref_ds.width
    rows = np.arange(win.row_off, win.row_off + win.height, dtype=np.int32)
    cols = np.arange(win.col_off, win.col_off + win.width, dtype=np.int32)
    rr, cc = np.meshgrid(rows, cols, indexing="ij")

    pixel_id = (rr.astype(np.int64) * np.int64(W) + cc.astype(np.int64)).ravel()
    xs, ys = rasterio.transform.xy(ref_ds.transform, rr, cc, offset="center")
    lon = np.asarray(xs, dtype=np.float64).ravel()
    lat = np.asarray(ys, dtype=np.float64).ravel()
    return rr.ravel(), cc.ravel(), lon, lat, pixel_id

def read_aligned_window(src_ds, ref_ds, win, layer_name, is_categorical):
    # Fast path: already aligned
    if (src_ds.crs == ref_ds.crs and
        src_ds.transform == ref_ds.transform and
        src_ds.width == ref_ds.width and
        src_ds.height == ref_ds.height):
        arr = src_ds.read(1, window=win).astype(np.float32)
        nodata = src_ds.nodata
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)
        if layer_name in ZERO_AS_NODATA:
            arr = np.where(arr == 0, np.nan, arr)
        return arr

    # Reproject to ref window
    dst_transform = rasterio.windows.transform(win, ref_ds.transform)
    dst_h, dst_w = win.height, win.width
    dst = np.full((dst_h, dst_w), np.nan, dtype=np.float32)

    resamp = Resampling.nearest if is_categorical else Resampling.bilinear

    reproject(
        source=rasterio.band(src_ds, 1),
        destination=dst,
        src_transform=src_ds.transform,
        src_crs=src_ds.crs,
        dst_transform=dst_transform,
        dst_crs=ref_ds.crs,
        src_nodata=src_ds.nodata,
        dst_nodata=np.nan,
        resampling=resamp
    )

    if layer_name in ZERO_AS_NODATA:
        dst = np.where(dst == 0, np.nan, dst)

    return dst

print("Helper functions loaded.")


## 5) Preview file discovery for a sample year
This part of the code is used to check the sample output. Generative AI tool was used to generate this part of code

In [ ]:
SAMPLE_YEAR = list(YEARS)[0]
print("Sample year:", SAMPLE_YEAR)

for var, folder in folders.items():
    fp = find_year_tif(folder, SAMPLE_YEAR)
    print(f"{var}: {fp if fp else 'not found'}")


## 6) Run extraction
Generative AI tool was used to generate this part of code

In [ ]:
soil_ds = rasterio.open(SOIL_FILE)

if os.path.exists(OUT_CSV):
    os.remove(OUT_CSV)
    print("Removed existing OUT_CSV:", OUT_CSV)

first_write = True

for year in YEARS:
    print(f"\n=== Year {year} ===")

    year_files = {}
    for var, folder in folders.items():
        fp = find_year_tif(folder, year)
        if fp is None:
            raise FileNotFoundError(f"Missing {var} tif for year {year} in: {folder}")
        year_files[var] = fp

    with rasterio.open(year_files["AET"]) as ref:
        open_year = {v: rasterio.open(p) for v, p in year_files.items()}

        try:
            chunks = []
            win_count = 0
            inside_pix_total = 0

            for _, win in ref.block_windows(1):
                win_count += 1
                aet_masked = ref.read(1, window=win, masked=True)
                inside = (~aet_masked.mask).ravel()

                if not inside.any():
                    continue

                inside_pix_total += int(inside.sum())

                row, col, lon, lat, pixel_id = window_coords_pixelid(ref, win)

                data = {
                    "year": np.full(inside.sum(), year, dtype=np.int16),
                    "pixel_id": pixel_id[inside],
                    "row": row[inside],
                    "col": col[inside],
                    "lon": lon[inside],
                    "lat": lat[inside],
                }

                for var, ds in open_year.items():
                    arr = read_aligned_window(ds, ref, win, layer_name=var, is_categorical=(var in CATEGORICAL))
                    data[var] = arr.ravel()[inside]

                soil_arr = read_aligned_window(soil_ds, ref, win, layer_name="SOIL", is_categorical=True)
                data["SOIL"] = soil_arr.ravel()[inside]

                chunks.append(pd.DataFrame(data))

            print("Windows scanned:", win_count)
            print("Inside pixels processed:", inside_pix_total)

            if not chunks:
                print("No inside pixels found for year:", year)
                continue

            df_y = pd.concat(chunks, ignore_index=True)
            predictor_cols = list(year_files.keys()) + ["SOIL"]
            df_y = df_y.dropna(subset=predictor_cols, how="all")

            df_y.to_csv(OUT_CSV, index=False, mode="w" if first_write else "a", header=first_write)
            first_write = False

            print("Appended rows:", len(df_y))
            display(df_y.head(5))

        finally:
            for ds in open_year.values():
                ds.close()

soil_ds.close()

print("\nDONE. Saved:", OUT_CSV)


## 7) Verify output CSV 

In [ ]:
if os.path.exists(OUT_CSV):
    df = pd.read_csv(OUT_CSV, nrows=200000)
    print("Sample rows loaded:", len(df))
    print("Columns:", list(df.columns))
    display(df.head(10))
else:
    print("OUT_CSV not found yet. Run the extraction cell first.")
